In [0]:
%sql
create volume dev_ext.bronze.landing
comment 'This is a landing managed volume'
;

In [0]:
dbutils.fs.mkdirs("/Volumes/dev_ext/bronze/landing/input")

In [0]:
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv", "/Volumes/dev_ext/bronze/landing/input")

In [0]:
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv", "/Volumes/dev_ext/bronze/landing/input")

In [0]:
%sql
create table dev_ext.bronze.invoice_cp
;


In [0]:
%sql 

copy into dev_ext.bronze.invoice_cp
from "/Volumes/dev_ext/bronze/landing/input"
fileformat = CSV
pattern = '*.csv'
format_options(
    'mergeSchema' = 'true',
    'header' = 'true'

)
COPY_OPTIONS (
    'mergeSchema' = 'true'
)
;

In [0]:
%sql
select * from dev_ext.bronze.invoice_cp

In [0]:
%sql
describe extended dev_ext.bronze.invoice_cp

In [0]:
%sql
create table dev_ext.bronze.invoice_cp_alt (
  InvoiceNo string,
  StockCode string,
  Quantity double,
  _insert_date timestamp

)
;


In [0]:
%sql 

copy into dev_ext.bronze.invoice_cp_alt
from ( 
  select InvoiceNo, StockCode, cast(Quantity as double) Quantity, current_timestamp() _insert_date
  from
"/Volumes/dev_ext/bronze/landing/input"
)
fileformat = CSV
pattern = '*.csv'
format_options(
    'mergeSchema' = 'true',
    'header' = 'true'

)
;

In [0]:
%sql
select count(1) from dev_ext.bronze.invoice_cp_alt